# Ricci Finance v10.6 Lecture Notebook

Dynamic 3D Ricci-capital manifold for finance.

Visual encoding: x/y = stable topology, z = Ricci stress, node size = dollar-volume mass, edge width = capital transport, edge color = Ricci curvature, animation = rolling-window evolution.

In [ ]:
from helper import *
import pandas as pd


## Build real yfinance market data


In [ ]:
tickers = DEFAULT_TICKERS

# Real market data: raw Yahoo Close and Volume.
# This replaces make_demo_market_data(), which was only synthetic lecture data.
prices, volumes, dollar_volume = download_market_data(
    tickers=tickers,
    period='1y',
    interval='1d',
)

returns = prices_to_returns(prices).dropna(axis=1, how='all')
dollar_volume = dollar_volume.reindex(index=returns.index, columns=returns.columns).fillna(0.0)

print('Last price date:', prices.index[-1])
prices.tail()


## Rolling Ricci-capital frames

In [ ]:
frames, starts = build_rolling_frames(
    returns=returns, window_size=60, step=5, max_frames=30,
    graph_mode='knn+bridges', k_neighbors=3, min_corr=0.05, max_bridges=3,
    dollar_volume=dollar_volume, use_capital_weighting=True, capital_alpha=0.35,
)
feature_df = rolling_feature_table(frames)
feature_df.tail()


## HMM hidden regimes

Fit an unsupervised Gaussian HMM using Ricci geometry plus capital-flow features.


In [ ]:
hmm_df, regime_names = compute_hmm_regimes(
    frames=frames,
    returns=returns,
    starts=starts,
    window_size=60,
    n_components=3,
    forward_days=5,
    random_state=42,
)

print(regime_names)
hmm_df.tail()


In [ ]:
fig_hmm = plot_hmm_regimes(hmm_df)
fig_hmm.show()


## 3D dynamic manifold

Run this in Jupyter to get an interactive Plotly animation with Play/Pause and rotation.

In [ ]:
base_graph = build_base_graph_for_layout(frames, all_nodes=returns.columns)
positions_3d = compute_stable_layout_3d(base_graph, seed=42, layout_k=0.45)
fig = build_3d_ricci_capital_animation(frames, positions_3d, z_mode='ricci_stress')
fig.show()


## Surgery-risk direction, not actual surgery

In [ ]:
fd = frames[-1]
surgery_risk_direction_table(fd.G).head(20)
